In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [4]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir /
    "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Train: 32337
Validation: 10952
Test: 10840


In [5]:
from pathlib import Path

ssftt_dir = (
    PROJECT_ROOT
    / "third_party"
    / "SSFTT"
)

ssftt_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print(ssftt_dir)

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\third_party\SSFTT


In [6]:
import subprocess

repo_dir = ssftt_dir / "HSI_SSFTT"

if not repo_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/zgr6010/HSI_SSFTT.git",
            str(repo_dir),
        ],
        check=True,
    )

print("Repository:", repo_dir)

Repository: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\third_party\SSFTT\HSI_SSFTT


In [7]:
for p in repo_dir.rglob("*.py"):
    print(p.relative_to(repo_dir))

cls_SSFTT_IP\get_cls_map.py
cls_SSFTT_IP\IP_train.py
cls_SSFTT_IP\SSFTTnet.py


In [8]:
import sys

ssftt_code_dir = (
    PROJECT_ROOT
    / "third_party"
    / "SSFTT"
    / "HSI_SSFTT"
    / "cls_SSFTT_IP"
)

if str(ssftt_code_dir) not in sys.path:
    sys.path.insert(0, str(ssftt_code_dir))

import SSFTTnet

print(SSFTTnet)
print(dir(SSFTTnet))

<module 'SSFTTnet' from 'c:\\Users\\Dines\\Documents\\Codex\\2026-05-15\\files-mentioned-by-the-user-dataset0\\dense-forest-monitoring\\third_party\\SSFTT\\HSI_SSFTT\\cls_SSFTT_IP\\SSFTTnet.py'>
['Attention', 'F', 'LayerNormalize', 'MLP_Block', 'NUM_CLASS', 'PIL', 'Residual', 'SSFTTnet', 'Transformer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_weights_init', 'init', 'nn', 'rearrange', 'time', 'torch', 'torchvision']


In [9]:
import inspect

print(
    inspect.getsource(
        SSFTTnet
    )
)

import PIL
import time
import torch
import torchvision
import torch.nn.functional as F
from einops import rearrange
from torch import nn
import torch.nn.init as init



def _weights_init(m):
    classname = m.__class__.__name__
    #print(classname)
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv3d):
        init.kaiming_normal_(m.weight)

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

# 等于 PreNorm
class LayerNormalize(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

# 等于 FeedForward
class MLP_Block(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GEL

In [10]:
print(
    inspect.signature(
        SSFTTnet.SSFTTnet
    )
)

(in_channels=1, num_classes=16, num_tokens=4, dim=64, depth=1, heads=8, mlp_dim=8, dropout=0.1, emb_dropout=0.1)


In [11]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [12]:
train_dataset = SalinasPatchDataset(
    scene=scene,
    indices=train_indices,
    patch_size=15,
    normalize=True,
)

val_dataset = SalinasPatchDataset(
    scene=scene,
    indices=val_indices,
    patch_size=15,
    normalize=True,
)

test_dataset = SalinasPatchDataset(
    scene=scene,
    indices=test_indices,
    patch_size=15,
    normalize=True,
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

32337 10952 10840


In [13]:
from sklearn.decomposition import PCA
import numpy as np
import joblib

# Number of spectral vectors used to fit PCA
MAX_PCA_SAMPLES = 200_000

rng = np.random.default_rng(42)

n_train = len(train_dataset)

# Randomly select training PATCHES only
n_selected = min(n_train, 2000)

selected_indices = rng.choice(
    n_train,
    size=n_selected,
    replace=False,
)

print("Training patches available:", n_train)
print("Patches sampled for PCA:", n_selected)

Training patches available: 32337
Patches sampled for PCA: 2000


In [14]:
sampled_patches = []

for idx in selected_indices:

    patch, _ = train_dataset[idx]

    # [204, 15, 15]
    sampled_patches.append(patch)

sampled_patches = np.stack(
    sampled_patches,
    axis=0,
)

print(
    "Sampled patch array:",
    sampled_patches.shape,
)

Sampled patch array: (2000, 204, 15, 15)


In [15]:
sampled_spectra = (
    sampled_patches
    .transpose(0, 2, 3, 1)
    .reshape(-1, 204)
)

print(
    "Spectral samples:",
    sampled_spectra.shape,
)

Spectral samples: (450000, 204)


In [16]:
if len(sampled_spectra) > MAX_PCA_SAMPLES:

    selected_spectra = rng.choice(
        len(sampled_spectra),
        size=MAX_PCA_SAMPLES,
        replace=False,
    )

    sampled_spectra = sampled_spectra[
        selected_spectra
    ]

print(
    "Final PCA fitting samples:",
    sampled_spectra.shape,
)

Final PCA fitting samples: (200000, 204)


In [17]:
ssftt_pca = PCA(
    n_components=30,
    svd_solver="randomized",
    random_state=42,
)

ssftt_pca.fit(
    sampled_spectra
)

explained_variance = (
    ssftt_pca
    .explained_variance_ratio_
    .sum()
)

print(
    f"30-component explained variance: "
    f"{explained_variance:.4f}"
)

30-component explained variance: 0.9996


In [18]:
del sampled_patches
del sampled_spectra

import gc
gc.collect()

20

In [19]:
pca_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "ssftt_30band_pca.joblib"
)

joblib.dump(
    ssftt_pca,
    pca_path,
)

print("Saved:", pca_path)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\ssftt_30band_pca.joblib


In [20]:
def ssftt_pca_transform(patch, pca):
    """
    Input:
        patch: [204, H, W]

    Output:
        [30, H, W]
    """

    bands, H, W = patch.shape

    spectra = (
        patch
        .transpose(1, 2, 0)
        .reshape(-1, bands)
    )

    reduced = pca.transform(spectra)

    reduced = reduced.reshape(
        H,
        W,
        30,
    )

    return reduced.transpose(
        2,
        0,
        1,
    ).astype(np.float32)

In [21]:
sample_patch, sample_label = train_dataset[0]

print("Original:", sample_patch.shape)

reduced_patch = ssftt_pca_transform(
    sample_patch,
    ssftt_pca,
)

print("PCA reduced:", reduced_patch.shape)

Original: (204, 15, 15)
PCA reduced: (30, 15, 15)


In [22]:
class SSFTTPCADataset(torch.utils.data.Dataset):

    def __init__(self, base_dataset, pca):
        self.base_dataset = base_dataset
        self.pca = pca

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        patch, label = self.base_dataset[idx]

        # [204, 15, 15] -> [30, 15, 15]
        patch = ssftt_pca_transform(
            patch,
            self.pca,
        )

        # Official SSFTT expects:
        # [B, 1, 30, H, W]
        patch = np.expand_dims(
            patch,
            axis=0,
        )

        return (
            torch.from_numpy(patch.astype(np.float32)),
            torch.tensor(label, dtype=torch.long),
        )

In [23]:
ssftt_train_dataset = SSFTTPCADataset(
    train_dataset,
    ssftt_pca,
)

ssftt_val_dataset = SSFTTPCADataset(
    val_dataset,
    ssftt_pca,
)

ssftt_test_dataset = SSFTTPCADataset(
    test_dataset,
    ssftt_pca,
)

In [24]:
BATCH_SIZE = 128

ssftt_train_loader = DataLoader(
    ssftt_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

ssftt_val_loader = DataLoader(
    ssftt_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

ssftt_test_loader = DataLoader(
    ssftt_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [25]:
x, y = next(iter(ssftt_train_loader))

print("Input :", x.shape)
print("Labels:", y.shape)

Input : torch.Size([128, 1, 30, 15, 15])
Labels: torch.Size([128])


In [26]:
class SSFTT204Dataset(torch.utils.data.Dataset):

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):

        patch, label = self.base_dataset[idx]

        # [204, 15, 15]
        # -> [1, 204, 15, 15]
        patch = np.expand_dims(
            patch,
            axis=0,
        ).astype(np.float32)

        return (
            torch.from_numpy(patch),
            torch.tensor(label, dtype=torch.long),
        )

In [27]:
ssftt204_train_dataset = SSFTT204Dataset(
    train_dataset,
)

ssftt204_val_dataset = SSFTT204Dataset(
    val_dataset,
)

ssftt204_test_dataset = SSFTT204Dataset(
    test_dataset,
)

In [28]:
ssftt204_train_loader = DataLoader(
    ssftt204_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

ssftt204_val_loader = DataLoader(
    ssftt204_val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

ssftt204_test_loader = DataLoader(
    ssftt204_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [29]:
from SSFTTnet import SSFTTnet

In [30]:
class SSFTT204(nn.Module):

    def __init__(
        self,
        num_classes=16,
        num_tokens=4,
        dim=64,
        depth=1,
        heads=8,
        mlp_dim=8,
        dropout=0.1,
        emb_dropout=0.1,
    ):
        super().__init__()

        self.spectral_projection = nn.Linear(
            204,
            30,
        )

        self.ssftt = SSFTTnet(
            in_channels=1,
            num_classes=num_classes,
            num_tokens=num_tokens,
            dim=dim,
            depth=depth,
            heads=heads,
            mlp_dim=mlp_dim,
            dropout=dropout,
            emb_dropout=emb_dropout,
        )

    def forward(self, x):

        # [B, 1, 204, H, W]

        x = x.squeeze(1)

        # [B, 204, H, W]
        x = x.permute(0, 2, 3, 1)

        # [B, H, W, 204]
        x = self.spectral_projection(x)

        # [B, H, W, 30]
        x = x.permute(0, 3, 1, 2)

        # [B, 30, H, W]
        x = x.unsqueeze(1)

        # [B, 1, 30, H, W]
        return self.ssftt(x)

In [31]:
ssftt_model = SSFTTnet(
    in_channels=1,
    num_classes=16,
    num_tokens=4,
    dim=64,
    depth=1,
    heads=8,
    mlp_dim=8,
    dropout=0.1,
    emb_dropout=0.1,
).to(device)

In [32]:
ssftt204_model = SSFTT204(
    num_classes=16,
    num_tokens=4,
    dim=64,
    depth=1,
    heads=8,
    mlp_dim=8,
    dropout=0.1,
    emb_dropout=0.1,
).to(device)

In [33]:
def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

In [34]:
print(
    "Official SSFTT parameters:",
    count_trainable_parameters(ssftt_model),
)

print(
    "SSFTT-204 parameters:",
    count_trainable_parameters(ssftt204_model),
)

Official SSFTT parameters: 153224
SSFTT-204 parameters: 159374


In [35]:
x, y = next(iter(ssftt_train_loader))

x = x.to(
    device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits = ssftt_model(x)

print("Official SSFTT input :", x.shape)
print("Official SSFTT output:", logits.shape)

Official SSFTT input : torch.Size([128, 1, 30, 15, 15])
Official SSFTT output: torch.Size([128, 16])


In [36]:
x204, y204 = next(iter(ssftt204_train_loader))

x204 = x204.to(
    device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits204 = ssftt204_model(x204)

print("SSFTT-204 input :", x204.shape)
print("SSFTT-204 output:", logits204.shape)

SSFTT-204 input : torch.Size([128, 1, 204, 15, 15])
SSFTT-204 output: torch.Size([128, 16])


In [37]:
def train_ssftt_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    running_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):

        x = x.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)

        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total

In [38]:
def evaluate_ssftt(
    model,
    loader,
    device,
):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in tqdm(loader, leave=False):

            x = x.to(
                device,
                dtype=torch.float32,
            )

            logits = model(x)

            preds = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            y_pred.extend(preds)
            y_true.extend(y.numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    return (
        accuracy,
        macro_f1,
        y_true,
        y_pred,
    )

In [39]:
from copy import deepcopy

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    ssftt_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

EPOCHS = 50

best_ssftt_val_f1 = -1.0
best_ssftt_epoch = 0
best_ssftt_state = None

ssftt_history = []

for epoch in range(1, EPOCHS + 1):

    train_loss = train_ssftt_one_epoch(
        ssftt_model,
        ssftt_train_loader,
        criterion,
        optimizer,
        device,
    )

    val_acc, val_f1, _, _ = evaluate_ssftt(
        ssftt_model,
        ssftt_val_loader,
        device,
    )

    scheduler.step()

    ssftt_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    if val_f1 > best_ssftt_val_f1:

        best_ssftt_val_f1 = val_f1
        best_ssftt_epoch = epoch

        best_ssftt_state = deepcopy(
            ssftt_model.state_dict()
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

print()
print("Best epoch:", best_ssftt_epoch)
print(
    "Best validation Macro-F1:",
    best_ssftt_val_f1,
)

Epoch 01/50 | Loss: 0.1333 | Val Acc: 0.8740 | Val Macro-F1: 0.9224


Epoch 02/50 | Loss: 0.0030 | Val Acc: 0.8836 | Val Macro-F1: 0.9228


Epoch 03/50 | Loss: 0.0022 | Val Acc: 0.8910 | Val Macro-F1: 0.9028


Epoch 04/50 | Loss: 0.0027 | Val Acc: 0.8961 | Val Macro-F1: 0.9143


Epoch 05/50 | Loss: 0.0009 | Val Acc: 0.8929 | Val Macro-F1: 0.8988


...

Epoch 46/50 | Loss: 0.0000 | Val Acc: 0.9182 | Val Macro-F1: 0.9564


Epoch 47/50 | Loss: 0.0000 | Val Acc: 0.9190 | Val Macro-F1: 0.9553


Epoch 48/50 | Loss: 0.0000 | Val Acc: 0.9165 | Val Macro-F1: 0.9556


Epoch 49/50 | Loss: 0.0000 | Val Acc: 0.9234 | Val Macro-F1: 0.9594


Epoch 50/50 | Loss: 0.0000 | Val Acc: 0.9139 | Val Macro-F1: 0.9548

Best epoch: 6
Best validation Macro-F1: 0.9615275405625092


In [40]:
ssftt_model.load_state_dict(
    best_ssftt_state
)

ssftt_model.eval()

SSFTTnet(
  (conv3d_features): Sequential(
    (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1))
    (1): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (conv2d_features): Sequential(
    (0): Conv2d(224, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0): ModuleList(
        (0): Residual(
          (fn): LayerNormalize(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=64, out_features=192, bias=True)
              (nn1): Linear(in_features=64, out_features=64, bias=True)
              (do1): Dropout(p=0.1, inplace=False)
            )
          )
        )
        (1): Residual(
          (fn): LayerNormalize(
            (no

In [41]:
(
    ssftt_test_acc,
    ssftt_test_f1,
    y_test_ssftt,
    y_pred_ssftt,
) = evaluate_ssftt(
    ssftt_model,
    ssftt_test_loader,
    device,
)

print("=" * 60)
print("OFFICIAL SSFTT SPATIAL TEST RESULTS")
print("=" * 60)
print(f"Accuracy : {ssftt_test_acc:.4f}")
print(f"Macro-F1 : {ssftt_test_f1:.4f}")

OFFICIAL SSFTT SPATIAL TEST RESULTS
Accuracy : 0.9815
Macro-F1 : 0.9541


In [42]:
ssftt_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "ssftt_official_spatial_best.pt"
)

torch.save(
    {
        "model_state_dict":
            ssftt_model.state_dict(),

        "best_epoch":
            best_ssftt_epoch,

        "best_val_macro_f1":
            best_ssftt_val_f1,

        "class_names":
            SALINAS_CLASS_NAMES,

        "patch_size":
            15,

        "original_bands":
            204,

        "input_components":
            30,

        "pca_path":
            str(pca_path),

        "seed":
            42,
    },
    ssftt_path,
)

print("Saved:", ssftt_path)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\ssftt_official_spatial_best.pt


In [43]:
ssftt_history_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "ssftt_official_spatial_history.csv"
)

pd.DataFrame(ssftt_history).to_csv(
    ssftt_history_path,
    index=False,
)

print(
    "Saved:",
    ssftt_history_path,
)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\ssftt_official_spatial_history.csv


In [44]:
import os
import time
import numpy as np
import pandas as pd
import torch

In [45]:
ssftt_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "ssftt_official_spatial_best.pt"
)

print("Checkpoint:", ssftt_path)
print("Exists:", ssftt_path.exists())

Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\ssftt_official_spatial_best.pt
Exists: True


In [46]:
checkpoint = torch.load(
    ssftt_path,
    map_location=device,
)

ssftt_model.load_state_dict(
    checkpoint["model_state_dict"]
)

ssftt_model = ssftt_model.to(device)
ssftt_model.eval()

print(
    "Loaded best epoch:",
    checkpoint["best_epoch"]
)

print(
    "Best validation Macro-F1:",
    checkpoint["best_val_macro_f1"]
)

Loaded best epoch: 6
Best validation Macro-F1: 0.9615275405625092


C:\Users\Dines\AppData\Local\Temp\ipykernel_36556\1302447729.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [47]:
param_count = sum(
    p.numel()
    for p in ssftt_model.parameters()
    if p.requires_grad
)

model_size_mb = (
    os.path.getsize(ssftt_path)
    / (1024 ** 2)
)

print(f"Parameters : {param_count:,}")
print(f"Parameters M: {param_count / 1e6:.6f}")
print(f"Model size : {model_size_mb:.3f} MB")

Parameters : 153,224
Parameters M: 0.153224
Model size : 0.598 MB


In [48]:
try:
    from thop import profile
except ImportError:
    print("Install THOP with:")
    print("%pip install thop")

In [49]:
example_x, _ = next(
    iter(ssftt_test_loader)
)

example_x = example_x[:1].to(
    device,
    dtype=torch.float32,
)

ssftt_model.eval()

with torch.no_grad():
    flops, params_thop = profile(
        ssftt_model,
        inputs=(example_x,),
        verbose=False,
    )

print(f"FLOPs  : {flops:,.0f}")
print(f"GFLOPs : {flops / 1e9:.6f}")

FLOPs  : 16,907,040
GFLOPs : 0.016907


In [50]:
if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    with torch.no_grad():
        _ = ssftt_model(example_x)

    torch.cuda.synchronize()

    peak_memory_mb = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 2)
    )

else:
    peak_memory_mb = float("nan")

print(
    f"Peak GPU memory: "
    f"{peak_memory_mb:.2f} MB"
)

Peak GPU memory: 47.74 MB


In [51]:
benchmark_batch, _ = next(
    iter(ssftt_test_loader)
)

benchmark_batch = benchmark_batch.to(
    device,
    dtype=torch.float32,
)

BATCH_SIZE_BENCH = benchmark_batch.shape[0]

print(
    "Benchmark batch size:",
    BATCH_SIZE_BENCH,
)

Benchmark batch size: 128


In [52]:
WARMUP = 30
REPEATS = 100

with torch.no_grad():

    for _ in range(WARMUP):

        _ = ssftt_model(
            benchmark_batch
        )

    torch.cuda.synchronize()

In [53]:
latencies_ms = []

with torch.no_grad():

    for _ in range(REPEATS):

        torch.cuda.synchronize()

        start = time.perf_counter()

        _ = ssftt_model(
            benchmark_batch
        )

        torch.cuda.synchronize()

        end = time.perf_counter()

        latencies_ms.append(
            (end - start) * 1000
        )

latencies_ms = np.asarray(
    latencies_ms
)

In [54]:
mean_batch_latency = (
    latencies_ms.mean()
)

median_batch_latency = (
    np.median(latencies_ms)
)

p95_batch_latency = (
    np.percentile(
        latencies_ms,
        95,
    )
)

std_batch_latency = (
    latencies_ms.std(
        ddof=1
    )
)

mean_sample_latency = (
    mean_batch_latency
    / BATCH_SIZE_BENCH
)

throughput = (
    BATCH_SIZE_BENCH
    / (mean_batch_latency / 1000)
)

print(
    f"Mean        : {mean_batch_latency:.3f} ms"
)

print(
    f"Median      : {median_batch_latency:.3f} ms"
)

print(
    f"P95         : {p95_batch_latency:.3f} ms"
)

print(
    f"Std         : {std_batch_latency:.3f} ms"
)

print(
    f"Per sample  : {mean_sample_latency:.4f} ms"
)

print(
    f"Throughput  : {throughput:.2f} samples/s"
)

Mean        : 8.623 ms
Median      : 5.602 ms
P95         : 10.491 ms
Std         : 13.052 ms
Per sample  : 0.0674 ms
Throughput  : 14844.05 samples/s


In [55]:
ssftt_efficiency = pd.DataFrame([
    {
        "model": "SSFTT (Official)",

        "accuracy": ssftt_test_acc,
        "macro_f1": ssftt_test_f1,

        "parameters": param_count,
        "parameters_m": param_count / 1e6,

        "model_size_mb": model_size_mb,

        "flops": flops,
        "gflops": flops / 1e9,

        "peak_gpu_memory_mb": peak_memory_mb,

        "batch_size": BATCH_SIZE_BENCH,

        "latency_batch_ms":
            mean_batch_latency,

        "latency_batch_median_ms":
            median_batch_latency,

        "latency_batch_p95_ms":
            p95_batch_latency,

        "latency_batch_std_ms":
            std_batch_latency,

        "latency_per_sample_ms":
            mean_sample_latency,

        "throughput_samples_sec":
            throughput,
    }
])

ssftt_efficiency

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SSFTT (Official),0.981458,0.95408,153224,0.153224,0.597601,16907040.0,0.016907,47.739746,128,8.622985,5.60225,10.49074,13.051519,0.067367,14844.047623


In [56]:
ssftt_efficiency_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "ssftt_official_efficiency.csv"
)

ssftt_efficiency.to_csv(
    ssftt_efficiency_path,
    index=False,
)

print(
    "Saved:",
    ssftt_efficiency_path
)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\ssftt_official_efficiency.csv


In [57]:
from pathlib import Path
import pandas as pd

salinas_out = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
)

hybrid_path = (
    salinas_out
    / "hybrid_spatial_efficiency.csv"
)

spectralformer_path = (
    salinas_out
    / "spectralformer_official_spatial_efficiency.csv"
)

ssftt_path = (
    salinas_out
    / "ssftt_official_efficiency.csv"
)

print("Hybrid:", hybrid_path.exists(), hybrid_path)
print("SpectralFormer:", spectralformer_path.exists(), spectralformer_path)
print("SSFTT:", ssftt_path.exists(), ssftt_path)

Hybrid: True c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\hybrid_spatial_efficiency.csv
SpectralFormer: True c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\spectralformer_official_spatial_efficiency.csv
SSFTT: True c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\ssftt_official_efficiency.csv


In [58]:
hybrid_df = pd.read_csv(hybrid_path)
spectralformer_df = pd.read_csv(spectralformer_path)
ssftt_df = pd.read_csv(ssftt_path)

print("Hybrid")
display(hybrid_df)

print("SpectralFormer")
display(spectralformer_df)

print("SSFTT")
display(ssftt_df)

Hybrid


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,Hybrid Spatial-Spectral,0.980996,0.963905,605712,0.605712,2.330137,82053120.0,0.082053,252.453613,128,43.81523,44.8742,45.74441,2.001661,0.342306,2921.358623


SpectralFormer


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.927399,0.918674,399381,0.399381,1.552929,43319680.0,0.04332,370.256348,128,67.640807,65.25065,71.43573,18.100158,0.528444,1892.348801


SSFTT


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SSFTT (Official),0.981458,0.95408,153224,0.153224,0.597601,16907040.0,0.016907,47.739746,128,8.622985,5.60225,10.49074,13.051519,0.067367,14844.047623


In [59]:
sota_comparison = pd.concat(
    [
        hybrid_df,
        spectralformer_df,
        ssftt_df,
    ],
    ignore_index=True,
)

sota_comparison

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,Hybrid Spatial-Spectral,0.980996,0.963905,605712,0.605712,2.330137,82053120.0,0.082053,252.453613,128,43.815230,44.87420,45.74441,2.001661,0.342306,2921.358623
1,SpectralFormer (Official),0.927399,0.918674,399381,0.399381,1.552929,43319680.0,0.043320,370.256348,128,67.640807,65.25065,71.43573,18.100158,0.528444,1892.348801
2,SSFTT (Official),0.981458,0.954080,153224,0.153224,0.597601,16907040.0,0.016907,47.739746,128,8.622985,5.60225,10.49074,13.051519,0.067367,14844.047623


In [60]:
comparison_columns = [
    "model",
    "accuracy",
    "macro_f1",
    "parameters",
    "parameters_m",
    "model_size_mb",
    "flops",
    "gflops",
    "peak_gpu_memory_mb",
    "batch_size",
    "latency_batch_ms",
    "latency_batch_median_ms",
    "latency_batch_p95_ms",
    "latency_batch_std_ms",
    "latency_per_sample_ms",
    "throughput_samples_sec",
]

sota_comparison = sota_comparison[
    [
        c
        for c in comparison_columns
        if c in sota_comparison.columns
    ]
]

display(
    sota_comparison.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "parameters_m": "{:.4f}",
        "model_size_mb": "{:.3f}",
        "gflops": "{:.4f}",
        "peak_gpu_memory_mb": "{:.2f}",
        "latency_batch_ms": "{:.3f}",
        "latency_batch_median_ms": "{:.3f}",
        "latency_batch_p95_ms": "{:.3f}",
        "latency_batch_std_ms": "{:.3f}",
        "latency_per_sample_ms": "{:.4f}",
        "throughput_samples_sec": "{:.2f}",
    })
)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,Hybrid Spatial-Spectral,0.9810,0.9639,605712,0.6057,2.330,82053120.000000,0.0821,252.45,128,43.815,44.874,45.744,2.002,0.3423,2921.36
1,SpectralFormer (Official),0.9274,0.9187,399381,0.3994,1.553,43319680.000000,0.0433,370.26,128,67.641,65.251,71.436,18.100,0.5284,1892.35
2,SSFTT (Official),0.9815,0.9541,153224,0.1532,0.598,16907040.000000,0.0169,47.74,128,8.623,5.602,10.491,13.052,0.0674,14844.05


In [61]:
hybrid_row = sota_comparison[
    sota_comparison["model"]
    == "Hybrid Spatial-Spectral"
].iloc[0]

spectral_row = sota_comparison[
    sota_comparison["model"]
    == "SpectralFormer (Official)"
].iloc[0]

ssftt_row = sota_comparison[
    sota_comparison["model"]
    == "SSFTT (Official)"
].iloc[0]

In [62]:
for name, row in [
    ("SpectralFormer", spectral_row),
    ("SSFTT", ssftt_row),
]:

    acc_gain = (
        hybrid_row["accuracy"]
        - row["accuracy"]
    ) * 100

    f1_gain = (
        hybrid_row["macro_f1"]
        - row["macro_f1"]
    ) * 100

    print(
        f"Hybrid vs {name}:"
    )

    print(
        f"  Accuracy gain : {acc_gain:+.2f} pp"
    )

    print(
        f"  Macro-F1 gain : {f1_gain:+.2f} pp"
    )

    print()

Hybrid vs SpectralFormer:
  Accuracy gain : +5.36 pp
  Macro-F1 gain : +4.52 pp

Hybrid vs SSFTT:
  Accuracy gain : -0.05 pp
  Macro-F1 gain : +0.98 pp



In [63]:
for name, row in [
    ("SpectralFormer", spectral_row),
    ("SSFTT", ssftt_row),
]:

    memory_ratio = (
        hybrid_row["peak_gpu_memory_mb"]
        / row["peak_gpu_memory_mb"]
    )

    latency_ratio = (
        hybrid_row["latency_per_sample_ms"]
        / row["latency_per_sample_ms"]
    )

    throughput_ratio = (
        hybrid_row["throughput_samples_sec"]
        / row["throughput_samples_sec"]
    )

    print(f"Hybrid vs {name}")
    print(
        f"  Memory ratio      : {memory_ratio:.2f}×"
    )
    print(
        f"  Latency ratio     : {latency_ratio:.2f}×"
    )
    print(
        f"  Throughput ratio  : {throughput_ratio:.2f}×"
    )
    print()

Hybrid vs SpectralFormer
  Memory ratio      : 0.68×
  Latency ratio     : 0.65×
  Throughput ratio  : 1.54×

Hybrid vs SSFTT
  Memory ratio      : 5.29×
  Latency ratio     : 5.08×
  Throughput ratio  : 0.20×



In [64]:
final_sota_path = (
    salinas_out
    / "sota_comparison_salinas_spatial.csv"
)

sota_comparison.to_csv(
    final_sota_path,
    index=False,
)

print(
    "Saved:",
    final_sota_path
)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\outputs\salinas\sota_comparison_salinas_spatial.csv
